# Demo 02 — The integration standard: MCP

In demo_01 we **hand-wired** two tools into the agent. Now imagine 50 tools across 3 apps and 3
models — that's the **N×M** glue problem. **MCP (Model Context Protocol)** collapses it to **N+M**:
each tool and each model implements *one* protocol. Think **USB-C for AI**, or — for this room —
**LSP for IDEs**: one protocol so any client talks to any server.

The same two tools (`get_order`, `get_policy`) now live in a standalone **MCP server**
(`../mcp_server.py`). The agent *discovers* them at runtime instead of having them hard-coded.
Watch the trace: you'll see the **handshake → discovery → call** sequence on the wire.

## Setup

In [1]:
# --- Setup: env, MLflow tracing, and a model factory -------------------------
import os
import nest_asyncio

from dotenv import load_dotenv

import mlflow
import mlflow.pydantic_ai

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.models.test import TestModel

load_dotenv()                        # reads ../.env
nest_asyncio.apply()                 # lets `await` work inside Jupyter cells

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT", "session7-app-dev"))
mlflow.pydantic_ai.autolog()              # type: ignore # every agent/LLM/tool/MCP call -> a trace span
mlflow.tracing.disable_notebook_display() # type: ignore # no inline trace widgets; we print via show_trace()

USE_TEST_MODEL = os.getenv("USE_TEST_MODEL", "0") == "1"

def make_model():
    # Live model MUST support tool/function calling (qwen2.5:7b / llama3.1:8b or larger).
    if USE_TEST_MODEL:
        return TestModel()
    return OpenAIChatModel(
        os.environ.get("MODEL", "qwen2.5:7b"),
        provider=OpenAIProvider(
            base_url=os.environ.get("OPENAI_BASE_URL", "http://localhost:11434/v1"),
            api_key=os.environ.get("OPENAI_API_KEY", "not-needed-for-local"),
        ),
    )

def show_trace():
    mlflow.flush_trace_async_logging()
    tr = mlflow.get_trace(mlflow.get_last_active_trace_id())
    for s in tr.data.spans:
        print(f"  [{s.span_type:5}] {s.name}")
    print("Open the timeline:", mlflow.get_tracking_uri())

print("setup ready | USE_TEST_MODEL =", USE_TEST_MODEL)


setup ready | USE_TEST_MODEL = False


## Connect the agent to the MCP server

`MCPServerStdio` launches `mcp_server.py` as a subprocess and speaks JSON-RPC to it over stdio.
On entering `async with agent`, the client performs the **`initialize` handshake** (capability
negotiation), then calls **`tools/list`** (discovery — "read the menu"). During the run it issues
**`tools/call`** for each tool the model picks. We never told the agent what tools exist — it
*discovered* them.

In [2]:
import sys
from pydantic_ai.mcp import MCPServerStdio

# Use this notebook's own interpreter to run the server (so deps match)
server = MCPServerStdio(command=sys.executable, args=["../mcp_server.py"])

# Each MCP server is a *toolset*. The agent discovers its tools at runtime — we never
# hard-code them. (Current API: toolsets=[...] + `async with agent`. The older
# mcp_servers=[...] + agent.run_mcp_servers() is deprecated.)
agent = Agent(make_model(), toolsets=[server],
              instructions="You are an order-support assistant. Use the available tools for real facts.")

async def ask(q: str):
    async with agent:                  # handshake (initialize) + discovery (tools/list) happen here
        return await agent.run(q)

r = await ask("Where's order A7731, and can I return the opened earbuds?")
print(r.output)


Your order A7731 has been delivered on June 22, 2026. The item in your order is the Wireless Earbuds (SKU: AZ-4471), and it appears that they were opened.

According to our returns policy:
- Sealed items can be returned for a full refund within 30 days of delivery.
- Unsealed or opened items may only be exchanged within 14 days of delivery; no cash refund is available.

Since the Wireless Earbuds in your order have been opened, you will need to request an exchange rather than a return. Please contact customer support to proceed with the exchange process.


## The protocol, made visible

The MLflow trace shows the MCP lifecycle as named spans:

- **`MCPServerStdio.list_tools`** → discovery (`tools/list`) — *"there's the menu."*
- **`MCPServerStdio.call_tool`** → execution (`tools/call`) — *"there's the call."*

Point at them on screen — that's your analogy, proven, not asserted.

In [3]:
show_trace()

  [AGENT] Agent.run
  [TOOL ] MCPServerStdio.list_tools
  [LLM  ] InstrumentedModel.request
  [TOOL ] ToolManager.handle_call
  [TOOL ] MCPServerStdio.call_tool
  [TOOL ] ToolManager.handle_call
  [TOOL ] MCPServerStdio.call_tool
  [TOOL ] MCPServerStdio.list_tools
  [LLM  ] InstrumentedModel.request
Open the timeline: http://127.0.0.1:5001


## Same agent, your daily tool

GitHub Copilot's **agent mode is an MCP host** too. When it adds a server it literally *"performs a
handshake and queries the tool list"* — the same two steps you just saw. Tools are **disabled by
default**; you enable them and approve calls (Allow / Confirm). See `../copilot-runbook.md` for the
live VS Code walkthrough. The thing you reach for every day is this exact protocol.